# Introduction to LangChain

[LangChain](https://www.langchain.com/) is a popular framework that allows users to quickly build apps and pipelines around **L**arge **L**anguage **M**odels (LLMs). It can be used for chatbots, **G**enerative **Q**uestion-**A**nwering (GQA), summarization, and much more.

The core idea of the framework is to allow for _"chaining"_ together different components to support advanced use-cases around LLMs (see an illustration [here](https://miro.medium.com/v2/resize:fit:1400/1*jJE0uZTBadEYqe0hEXlWuQ.png)). Chains may consist of multiple components from several modules among which the main are the following:

* **Models**: Modern LLMs are typically accessed through a chat model interface that takes a list of messages as input and returns a message as output. The latest generation of chat models natively support **tool calling APIs**, which enable LLMs to interact with external services, APIs, and databases.

* **Prompt templates**: Prompt template is an object that takes user input and combines it with a template for a particular prompt type into the final string or message (prompt). There are different type of prompt templates, suitable for different kinds of models and different tasks.

* **Output parsers**: These are responsible for taking in the output of a model (strings or a message) and transforming it into a more usable form, such as JSON that matches a given schema.


In [1]:
!pip -q install langchain

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

import os
from google.colab import userdata

# Using large language models in LangChain

LangChain supports numerous large language model (LLM) providers, including both those that offer open and free models (such as Hugging Face) and those that offer proprietary models that require payment (e.g., OpenAI). The use of these different models in LangChain is almost identical, one only needs to instantiate a differnt class to use different models (e.g., ChatGroq vs ChatOpenAI).

We will explore the use of LangChain with open LLMs and all we do can be easily mapped to the work with proprietary models.

### Access to open models via Groq

We will use [Groq](https://groq.com/) API to access some of state-of-the-art open model, such as [GPT-OSS-20b](https://huggingface.co/openai/gpt-oss-20b) or [GPT-OSS-120b](https://huggingface.co/openai/gpt-oss-120b)

Groq is a company specialised in accelerating the inference process for pre-trained AI models, such as transformer-based models, using specialized hardware and software. It is providing a platform that allows for fast and  efficient running of AI models. It also offers certain number of calls free of charge. To learn about the number of free calls (not worth stating it here as it is prone to change), take a look at [this page](https://console.groq.com/docs/rate-limits). You can sign up [here](https://console.groq.com/keys) for an (free) API key, required to use Groq's API.

If working in Colab, use **Colab Secrets** to store your Groq API key. To learn how to do that, see, for example, [this short article](https://labs.thinktecture.com/secrets-in-google-colab-the-new-way-to-protect-api-keys/). If working offline, in DataSpell or a similar development environment, you are advised to store your Groq API key in the .env file.

In [2]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

GroqAPI can be accessed via LangChain, which allows for easy integration of an LLM into an overall application workflow. To make use of this integration, we first need to install the `langchain_groq` library:

In [3]:
!pip -q install langchain_groq

### Set up an LLM and create simple chains

Next we create a connection to an LLM instance, via Groq, by instantiating the ChatGroq class with a model supported by Groq (see the full list [here](https://console.groq.com/docs/models)):

In [13]:
from langchain_groq import ChatGroq

creative_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.9, # setting high temperature to "foster" creativity of the model
    max_tokens=500,
    model_kwargs={'seed':9} # making the generation process more determistic and increases the chance of reproducable results
    )

See the [ChatGroq API](https://reference.langchain.com/python/integrations/langchain_groq/?h=chatgroq#langchain_groq.ChatGroq) for the details on the constructor call, the available methods and the response formats

Note that LLMs we use are fine-tuned for chat, which means that the interaction with the model is typically structured as follows:
```
System message : You are a helpful and kind assistant that helps users make their travel plans.
Your suggestions are concise and to the point.

Human: I like big bustling cities, where should I go?

AI: You should go to Sydney, Australia

Human: Sounds good. What should I do when I'm there?

AI:
```

In [7]:
DEFAULT_SYSTEM_PROMPT = """
You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while never including any harmful, unethical, or illegal content.
If a question does not make any sense, or is not factually coherent, explain the issue with the question instead of giving an incorrect answer.
If you don't know the answer to a question, just say "I do not know".
"""

In [8]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", DEFAULT_SYSTEM_PROMPT),
        ("human", "I'm working on a marketing campaign and need your help. Specifically, I need your help with {question}"),
    ]
)

In [9]:
cookie_name_question = "finding a good name for a company that makes and sells hand-made healthy cookies"

print(prompt_template.format(question=cookie_name_question))

System: 
You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while never including any harmful, unethical, or illegal content.
If a question does not make any sense, or is not factually coherent, explain the issue with the question instead of giving an incorrect answer.
If you don't know the answer to a question, just say "I do not know".

Human: I'm working on a marketing campaign and need your help. Specifically, I need your help with finding a good name for a company that makes and sells hand-made healthy cookies


In [14]:
creative_question_chain = prompt_template | creative_llm

response = creative_question_chain.invoke({"question": cookie_name_question})

In [11]:
type(response)

langchain_core.messages.ai.AIMessage

Let's explore the response in more detail

In [15]:
display(response.content)

'Here are a few ideas that blend the “hand‑made” and “healthy” vibe with the deliciousness of cookies. Feel free to mix and match words, tweak spellings, or use them as a springboard for your own brand story.\n\n| # | Company Name | Why it works | Possible Tagline |\n|---|--------------|--------------|------------------|\n| 1 | **Wholesome Crumb Co.** | “Wholesome” signals health; “Crumb” feels familiar and tactile. | “Baked by hand, loved by all.” |\n| 2 | **Nature’s Oven** | Emphasizes natural ingredients and the warmth of an oven. | “Where nature meets comfort.” |\n| 3 | **Hand‑Baked Harvest** | Suggests fresh, seasonal ingredients and artisanal craft. | “Harvest‑to‑cookie goodness.” |\n| 4 | **Purely Pecan & Co.** | “Purely” hints at no artificial additives; can be any nut/fruit. | “Naturally nutty, irresistibly pure.” |\n| 5 | **Garden‑Glaze Cookies** | “Garden” evokes fresh produce; “Glaze” adds a touch of indulgence. | “From garden to bite.” |\n| 6 | **Heart‑Made Cookies** | Pla

Here are a few ideas that blend the “hand‑made” and “healthy” vibe with the deliciousness of cookies. Feel free to mix and match words, tweak spellings, or use them as a springboard for your own brand story.

| # | Company Name | Why it works | Possible Tagline |
|---|--------------|--------------|------------------|
| 1 | **Wholesome Crumb Co.** | “Wholesome” signals health; “Crumb” feels familiar and tactile. | “Baked by hand, loved by all.” |
| 2 | **Nature’s Oven** | Emphasizes natural ingredients and the warmth of an oven. | “Where nature meets comfort.” |
| 3 | **Hand‑Baked Harvest** | Suggests fresh, seasonal ingredients and artisanal craft. | “Harvest‑to‑cookie goodness.” |
| 4 | **Purely Pecan & Co.** | “Purely” hints at no artificial additives; can be any nut/fruit. | “Naturally nutty, irresistibly pure.” |
| 5 | **Garden‑Glaze Cookies** | “Garden” evokes fresh produce; “Glaze” adds a touch of indulgence. | “From garden to bite.” |
| 6 | **Heart‑Made Cookies** | Plays on “heart” for both health and love in baking. | “Baked with love, eaten with purpose.” |
| 7 | **Green‑Grain Cookie Co.** | Highlights whole grains and eco‑friendly focus. | “Whole grains, whole hearts.” |
| 8 | **Sprout & Crumb** | “Sprout” signals freshness; “Crumb” keeps it cookie‑centric.

In [32]:
from pprint import pprint

pprint(response.response_metadata)

{'finish_reason': 'length',
 'logprobs': None,
 'model_name': 'openai/gpt-oss-20b',
 'model_provider': 'groq',
 'service_tier': 'on_demand',
 'system_fingerprint': 'fp_c5a89987dc',
 'token_usage': {'completion_time': 0.622213183,
                 'completion_tokens': 500,
                 'completion_tokens_details': {'reasoning_tokens': 134},
                 'prompt_time': 0.010630138,
                 'prompt_tokens': 184,
                 'prompt_tokens_details': None,
                 'queue_time': 0.197571595,
                 'total_time': 0.632843321,
                 'total_tokens': 684}}


While not all of these metadata elements are relevant to us now, it would be useful to keep track of the tokens exchanged, as that is directly related to the pricing, that is, the use of the free quota.

In [16]:
def get_tokens_used(groq_response):
  token_usage = groq_response.response_metadata['token_usage']
  return {
      'prompt_tokens':token_usage['prompt_tokens'],
      'completion_tokens':token_usage['completion_tokens'],
      'reasoning_tokens':token_usage['completion_tokens_details']['reasoning_tokens'],
      'total_tokens':token_usage['total_tokens']
  }

get_tokens_used(response)

{'prompt_tokens': 184,
 'completion_tokens': 500,
 'reasoning_tokens': 134,
 'total_tokens': 684}

To make the interaction with an LLM more specific to the task at hand, we can alter the system prompt and provide more task-specific guidance in the user message:

In [18]:
branding_system_msg = """
You are a highly creative assistant, with a plenty of original ideas. You especially excell in inventing catchy brand and product promotional messages.
"""

branding_question = """
What would be a good name for a company that makes and sells {product}? Suggest {number} distinct names.
Structure the results as a python dictionary with the company name as the key and an explanation for the suggested name as the value.
Return only this dictionary and nothing else.
"""

branding_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", branding_system_msg),
        ("human", branding_question),
    ]
)

Build a simple chain using the above created prompt template and LLM

In [19]:
branding_chain = branding_prompt_template | creative_llm

branding_response = branding_chain.invoke({'product': 'colorful running gear', 'number':3})

In [23]:
print(branding_response.content)

{
    "Chromatic Strides": "A name that blends the idea of vibrant colors with the forward motion of running, suggesting gear that turns every step into a colorful statement.",
    "Vivid Velocity": "Combines energetic motion with bright, eye‑catching hues, conveying that the gear not only moves you fast but also makes a bold visual impact.",
    "Spectrum Sprint": "Highlights a full range of colors (spectrum) while emphasizing speed (sprint), ideal for a brand that offers diverse, high‑performance running apparel."
}


In [24]:
get_tokens_used(branding_response)

{'prompt_tokens': 158,
 'completion_tokens': 314,
 'reasoning_tokens': 194,
 'total_tokens': 472}

We can also request from the LLM to generate the output in the JSON format and then use LangChain's `JsonOutputParser` to parse the content so that it can be used further in the program:

In [25]:
branding_question_json = """
What would be a good name for a company that makes and sells {product}? Suggest {number} distinct names.
Structure the results as a JSON list of dictionaries, where each dictionary has the company name as its key and an explanation for the suggested name as its value.
Return only this JSON list and nothing else.
"""

branding_prompt_template_json = ChatPromptTemplate.from_messages(
    [
        ("system", branding_system_msg),
        ("human", branding_question_json),
    ]
)

branding_chain_json = branding_prompt_template_json | creative_llm | JsonOutputParser()

In [26]:
branding_json = branding_chain_json.invoke({'product': 'colorful running gear', 'number':4})

In [29]:
display(branding_json)

[{'ColorStride': 'Combines the vividness of color with the dynamic motion of running, implying a brand that energizes every step.'},
 {'PrismPace': "Suggests a spectrum of shades that catch the eye, while 'Pace' ties directly to the runner's rhythm."},
 {'ChromaticTrail': "Evokes bright, bold hues that illuminate a runner's path, positioning the gear as both functional and fashionable."},
 {'VividVelocity': 'Highlights lively, eye-catching designs that boost performance and confidence on every run.'}]

In [30]:
type(branding_json)

list

### Simple chain sequencing

We can also explore a simple sequantial chain:

In [33]:
# Chain 1
chain1_prompt = ChatPromptTemplate.from_messages([
    ("system", branding_system_msg),
    ("human", "Please suggest a catchy name for a company that makes and sells {product}. Return just the company name and nothing else.")
])

chain1 = chain1_prompt | creative_llm | StrOutputParser()

In [34]:
# Chain 2
chain2_prompt = ChatPromptTemplate.from_messages([
    ("system", branding_system_msg),
    ("human", "Please write a short promotional message for the following company: {company_name}.")
])

chain_seq = {"company_name": chain1} | chain2_prompt | creative_llm | StrOutputParser()

company_desc = chain_seq.invoke({"product":"hand-made healthy cookies"})

Note that in the `chain_seq` code above, the dictionary `{"company_name": chain1}` should be interpreted as the output of `chain1` is stored in the variable `company_name`

In [35]:
print(company_desc)

**CrumbWell:**  
Because every crumb matters.  
Indulge in our guilt‑free, wholesome crumb‑sized delights that turn every bite into a wellness win.  

Try CrumbWell today—where sweet meets healthy, one crumb at a time.


### Answering factual questions

We can also ask an LLM some factual questions. But, for that, we will need a new model instance, since to answer factual questions the model's playfulness (temperature) needs to be reduced to zero


In [36]:
factual_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    model_kwargs={'seed':9}
)

In [37]:
questions = [
    "What is the capital of Serbia?",
    "What is the largest city in Europe in terms of population?",
    "What movie got the highest number of the U.S. Academy Awards (aka Oscars)?",
    "Where the next Summer Olympic Games will be held?"
]

In [38]:
factual_questions_system_msg = """
Please answer the user's questions by giving just a direct, factual response and a short explanation for each response.
When responding, consider that we are currently in the 2nd week of November, 2026.
If you do not know the answer to a question, respond with 'I don't know'.
"""

questions_str = "\n".join([f"{i+1}) {q}" for i, q in enumerate(questions)])

In [39]:
factual_questions_prompt = ChatPromptTemplate.from_messages([
    ("system", factual_questions_system_msg),
    ("human", "QUESTIONS:\n{questions}")
])

print(factual_questions_prompt.format(questions=questions_str))

System: 
Please answer the user's questions by giving just a direct, factual response and a short explanation for each response.
When responding, consider that we are currently in the 2nd week of November, 2026. 
If you do not know the answer to a question, respond with 'I don't know'.

Human: QUESTIONS:
1) What is the capital of Serbia?
2) What is the largest city in Europe in terms of population?
3) What movie got the highest number of the U.S. Academy Awards (aka Oscars)?
4) Where the next Summer Olympic Games will be held?


In [40]:
factual_questions_chain = factual_questions_prompt | factual_llm

factual_responses = factual_questions_chain.invoke({'questions': questions_str})

In [41]:
print(factual_responses.content)

1. **Belgrade** – It is the capital and largest city of Serbia.  
2. **Istanbul, Turkey** – Its metropolitan area has over 15 million residents, making it the most populous city in Europe.  
3. **11 Oscars** – The record for most Academy Awards won by a single film is 11, shared by *Ben‑Hur* (1959), *Titanic* (1997), and *The Lord of the Rings: The Return of the King* (2003).  
4. **Los Angeles, USA** – The 2028 Summer Olympic Games are scheduled to be held there.


In [42]:
get_tokens_used(factual_responses)

{'prompt_tokens': 192,
 'completion_tokens': 430,
 'reasoning_tokens': 288,
 'total_tokens': 622}

We will now challenge the LLM with questions it should not be able to answer:

In [43]:
non_answerable_questions = "Who is the current president of the USA?\nWho is currently the best tennis player accoridng to the ATP list?"
factual_responses2 = factual_questions_chain.invoke({'questions': non_answerable_questions})
print(factual_responses2.content)

**Who is the current president of the USA?**  
I don't know.  
*Explanation:* The 2024 U.S. presidential election, which determines the president for 2025‑2029, has not yet taken place as of the current date in November 2026, so the outcome is unknown.

**Who is currently the best tennis player according to the ATP list?**  
I don't know.  
*Explanation:* The ATP rankings are updated weekly, and without up‑to‑date data from 2026, I cannot provide the current top‑ranked player.


### Instantiating and using proprietary LLMs: an OpenAI example

Let's try the same tasks with OpenAI's models. LangChain offers extensive support for working with OpenAI models, available through the `langchain_openai` package. For an overview, check [this page of the LangChain documentation](https://python.langchain.com/docs/integrations/providers/openai/).

We'll start by installing additional libraries and setting up a few additional prerequisites:

In [48]:
!pip install -qU langchain-core openai langchain-openai

To use OpenAI's generative models, we need to get our API key which can be retrieved by signing up for an account on the
[OpenAI's page for developers](https://platform.openai.com/docs/overview).

**Note**: Obtaining an OpenAI API key is free of charge. It is running of their models that is charged, so you can open an account and obtain the API key for free. However, you won't be able to run a model (or any LangChain component that integrates an OpenAI model) until you deposit some money on your OpenAI account. You may also want to check their [pricing sheme](https://platform.openai.com/docs/pricing) to get a better undersanding of how they charge for API calls.

Once you have your OpenAI API key, you can add it to the Colab Secrets or enter it in some other privacy protected way. Here, we use Colab Secrets to retrieve API key:

In [1]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

As above, we will access OpenAI's models via LangChain's ChatModel abstraction, since chat (= back and forth interaction with the user) is the interaction mode typical for the latest generation of LLMs. More precisely, this means that we will be using the [`ChatOpenAI` class](https://python.langchain.com/docs/integrations/chat/openai/) from the `langchain_openai` module.

As for the specific OpenAI model, we will use **gpt-4o-mini** since we will be working with text only and that model has the best price / performance ratio (as can be seen [here](https://platform.openai.com/docs/pricing))

In [4]:
from langchain_openai import ChatOpenAI

openai_creative = ChatOpenAI(model='gpt-4o-mini', temperature=0.9, seed=9, max_tokens=500)

We'll use the same simple questions as before with the gpt-oss-20b example:

In [7]:
branding_system_msg = """You are a highly creative assistant, with a plenty of original ideas. You especially excell in inventing catchy brand and product promotional messages."""
branding_question = """What would be a good name for a company that makes and sells {product}? Suggest {number} distinct names."""

messages = [
    ("system", branding_system_msg),
    ("human", branding_question)
]


branding_prompt_template = ChatPromptTemplate.from_messages(messages)

branding_openai_chain = branding_prompt_template | openai_creative

branding_openai_response = branding_openai_chain.invoke({'product':'colorful running gear', 'number':3})

In [8]:
type(branding_openai_response)

langchain_core.messages.ai.AIMessage

The response of OpenAI's GPT-4o-mini model has the same structure as the one we saw above for gpt-oss-20b model, that is, it includes the `content` and `response_metadata` components. The difference is in the metadata component, which is more complex in the case of GPT-4o-mini as it can work with multiple modalities.

In [9]:
print(branding_openai_response.content)

Sure! Here are three distinct names for a company that specializes in colorful running gear:

1. **VividStride** - Emphasizing both the vibrant colors and the dynamic motion of running, this name suggests a lively and energetic approach to fitness.

2. **ChromaticDash** - This name combines the idea of color (chromatic) with the speed and urgency of running (dash), perfect for a brand that celebrates both style and performance.

3. **RainbowRunner** - A playful and memorable name that evokes images of a spectrum of colors, making it clear that your brand is all about standing out on the track or trail. 

Let me know if you need more options or help with anything else!


In [10]:
from pprint import pprint

pprint(branding_openai_response.response_metadata)

{'finish_reason': 'stop',
 'id': 'chatcmpl-EhIAbS0Wjhm7QLh6ERCpHoXlUZnMw',
 'logprobs': None,
 'model_name': 'gpt-4o-mini-2024-07-18',
 'model_provider': 'openai',
 'service_tier': 'default',
 'system_fingerprint': 'fp_1dbdf820db',
 'token_usage': {'completion_tokens': 143,
                 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                               'audio_tokens': 0,
                                               'reasoning_tokens': 0,
                                               'rejected_prediction_tokens': 0,
                                               'text_tokens': None},
                 'prompt_tokens': 61,
                 'prompt_tokens_details': {'audio_tokens': 0,
                                           'cache_write_tokens': None,
                                           'cached_tokens': 0,
                                           'image_tokens': None,
                                           'text_tokens': Non

Let's now see how GPT-4o-mini will respond to factual questions.
For this task, we will set its temperature to zero, thus decreasing the chance for hallucinations and allowing for replicating the results.

In [13]:
openai_factual = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0
)

In [15]:
questions = [
    "What is the capital of Serbia?",
    "What is the largest city in Europe in terms of population?",
    "What movie got the highest number of the U.S. Academy Awards (aka Oscars)?",
    "Where the next Summer Olympic Games will be held?",
    "What is the longest commercial flight and between which cities?"
]

questions_str = "\n".join([f"{i+1}) {q}" for i, q in enumerate(questions)])

factual_system_msg = """
Please answer the questions given below, by giving a direct response followed by a short explanation.
When responding, consider that we are currently in the 2nd week of November, 2026.
If you do not know the answer, just say so.
"""

In [16]:
factual_questions_prompt = ChatPromptTemplate.from_messages([
    ("system", factual_system_msg),
    ("human", "QUESTIONS:\n{questions}")
])

factual_openai_chain = factual_questions_prompt | openai_factual | StrOutputParser()

factual_openai_responses = factual_openai_chain.invoke({"questions":questions_str})

In [ ]:
print(factual_openai_responses)

1) The capital of Serbia is Belgrade.  
Belgrade is the largest city in Serbia and serves as its political, cultural, and economic center.

2) The largest city in Europe in terms of population is Istanbul.  
Istanbul is a transcontinental city in Turkey, straddling Europe and Asia, and has a population that exceeds 15 million.

3) The movie that got the highest number of the U.S. Academy Awards is "Titanic," "The Lord of the Rings: The Return of the King," and "Ben-Hur," each winning 11 Oscars.  
These films are tied for the most Academy Awards won by a single film in Oscar history.

4) The next Winter Olympic Games will be held in Milan and Cortina d'Ampezzo, Italy, in 2026.  
These cities were selected to host the Winter Games, marking a return of the Olympics to Italy.

5) The longest commercial flight is between Singapore and New York City (JFK), operated by Singapore Airlines.  
This flight covers a distance of approximately 9,537 miles (15,349 kilometers) and can take around 18-1

In [17]:
# non_answerable_questions = "Who is the current president of the USA?\nWho is currently the best tennis player accoridng to the ATP list?"
difficult_factual = factual_openai_chain.invoke({'questions': non_answerable_questions})
print(difficult_factual)

I do not know the answer to either question. My training only includes data up to October 2023, and I cannot provide current information beyond that date.


### Choosing an LLM

The pace of large language model development and deprecation is extremely fast. Furthermore, there are so many different LLMs, both open and proprietary ones. So, a question that naturally arises is how to choose a model to use in a particular LLM-based application.

While there is no easy and straightforward answer to that question, what one can do is to:
* Consult one of the many LLM leaderboards that rank LLMs based on a variety of evalution tasks. Well known ones include [Chatbot Arena](https://lmarena.ai/leaderboard/) and [Vellum.ai](https://www.vellum.ai/llm-leaderboard)
* Keep track of models offered by [OpenAI](https://platform.openai.com/docs/models/overview), [Anthropic](https://platform.claude.com/docs/en/models/overview), and [Google](https://deepmind.google/models/), as the major providers of proprietary models
* read online available articles (from credible sources) offering a comparison of current LLMs